# Ottimizzazione Data Analytics

Una panoramica delle librerie per la manipolazione dati in Python

> **Struttura del notebook:** Le prime sezioni (fino al separatore) vengono trattate a lezione. Il resto è materiale di approfondimento per lo studio autonomo.

In [ ]:
# Setup delle dipendenze (eseguire solo se necessario)
#!pip install pyarrow pandas polars

## Pandas: Il Data Frame Standard

**Pandas** è diventato lo standard de facto per la manipolazione dei dati in Python, offrendo:

- Una sintassi intuitiva per la manipolazione di tabelle
- Funzioni per caricamento e salvataggio dati da vari formati
- Strumenti per pulizia, trasformazione e analisi dei dati
- Integrazione profonda con l'ecosistema scientifico Python

## I Limiti di Pandas

Con l'aumento delle dimensioni dei dati, Pandas presenta diverse limitazioni:

1. **Prestazioni**: implementato in Python/C, con overhead legato al GIL
2. **Memoria**: carica l'intero dataset in RAM
3. **Parallelismo**: limitato supporto per elaborazione parallela
4. **Tipo Dati**: gestione inefficiente di alcuni tipi (es. stringhe, liste)
5. **Scalabilità**: difficoltà con dataset che superano la RAM disponibile

## Pandas vs Polars

**Polars** è una libreria per DataFrame scritta in Rust, basata su Apache Arrow, che supera molti dei limiti di Pandas.

| **Pandas** | **Polars** |
|------------|------------|
| Indice Row-oriented | Column-oriented (Apache Arrow) |
| Operazioni in-place | Operazioni immutabili (funzionale) |
| GIL Python limita parallelismo | Computazioni parallele in Rust |
| NA, None, NaN (gestione inconsistente) | Solo `null` (gestione uniforme) |
| Indicizzazione con `.loc`/`.iloc` | Nessun indice, API basata su espressioni |
| Esecuzione eager | Supporto lazy evaluation nativo |

## Confronto pratico: creazione e tipi

Vediamo come si presentano le operazioni base nelle due librerie.

In [2]:
import pandas as pd
import polars as pl

# Configurazione per la visualizzazione
pd.set_option('display.max_rows', 6)
pd.set_option('max_colwidth', 15)
pl.Config().set_tbl_rows(6)

polars.config.Config

In [3]:
# Creazione DataFrame in Pandas
pandas_df = pd.DataFrame({
    'A': [1, 2, 3, 4, 5],
    'B': [10, 20, 30, 40, 50],
    'C': ['a', 'b', 'c', 'd', 'e']
})

# Creazione DataFrame in Polars
polars_df = pl.DataFrame({
    'A': [1, 2, 3, 4, 5],
    'B': [10, 20, 30, 40, 50],
    'C': ['a', 'b', 'c', 'd', 'e']
})

In [4]:
# Tipi dati in Pandas vs Polars
print("Pandas dtypes:")
print(pandas_df.dtypes)
print("\nPolars dtypes:")
print(polars_df.dtypes)

Pandas dtypes:
A    int64
B    int64
C      str
dtype: object

Polars dtypes:
[Int64, Int64, String]


Notare come Pandas usi `object` per le stringhe (un tipo generico e inefficiente), mentre Polars ha un tipo `String` nativo.

In [5]:
# I/O confronto: sintassi equivalente ma internamente molto diversa

# Pandas
pandas_df.to_csv("data.csv")
df_pandas = pd.read_csv("data.csv")

# Polars
polars_df.write_csv("data.csv")
df_polars = pl.read_csv("data.csv")

## Principali Differenze: Context e Expressions

Polars introduce due concetti fondamentali per manipolare i dati:

1. **Context**: il contesto in cui un'espressione viene valutata
2. **Expressions**: operazioni su colonne definite tramite API fluida

### I Tre Context di Polars:

1. **Selection**: `df.select(...)` e `df.with_columns(...)`
2. **Filtering**: `df.filter(...)`
3. **Aggregation**: `df.group_by(...).agg(...)`

Ogni contesto interpreta le expressions in modo diverso.

## Pandas vs Polars: Selezione Colonne

In [6]:
# Pandas: selezione colonne
pandas_df[['A', 'B']]  # Sintassi classica
# Alternativa: pandas_df.loc[:, ['A', 'B']]

,A,B
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


In [7]:
# Polars: selezione colonne
polars_df.select(['A', 'B'])  # Sintassi esplicita
# polars_df.select(pl.col(['A', 'B']))

A,B
i64,i64
1,10
2,20
3,30
4,40
5,50


## Pandas vs Polars: Filtri

In [8]:
# Pandas: filtri con operatori booleani
pandas_df[(pandas_df['A'] > 2) & (pandas_df['B'] <= 40)]
# Alternativa: pandas_df.query("A > 2 & B <= 40")

,A,B,C
2,3,30,c
3,4,40,d


In [9]:
# Polars: filtri con context dedicato
polars_df.filter(
    (pl.col("A") > 2) & (pl.col("B") <= 40)
)

A,B,C
i64,i64,str
3,30,"""c"""
4,40,"""d"""


## Pandas vs Polars: Aggregazioni

In [10]:
import numpy as np

# Dataset per le aggregazioni
df_agg = pd.DataFrame({
    'gruppo': ['A', 'A', 'B', 'B', 'A', 'C'],
    'valore1': [10, 20, 30, 25, 15, 40],
    'valore2': [100, 120, 200, 210, 110, 300]
})

In [11]:
# Pandas: aggregazione
df_agg.groupby('gruppo').agg({
    'valore1': ['sum', 'mean'],
    'valore2': ['min', 'max']
})

valore1       valore2     
           sum  mean     min  max
gruppo                           
A           45  15.0     100  120
B           55  27.5     200  210
C           40  40.0     300  300

In [12]:
# Polars: stesso dataset
pl_agg = pl.from_pandas(df_agg)

# Aggregazione con context dedicato
pl_agg.group_by('gruppo').agg([
    pl.col('valore1').sum().alias('valore1_sum'),
    pl.col('valore1').mean().alias('valore1_mean'),
    pl.col('valore2').min().alias('valore2_min'),
    pl.col('valore2').max().alias('valore2_max')
])

gruppo,valore1_sum,valore1_mean,valore2_min,valore2_max
str,i64,f64,i64,i64
"""B""",55,27.5,200,210
"""C""",40,40.0,300,300
"""A""",45,15.0,100,120


## Vantaggi Esclusivi di Polars

### 1. Gestione Nativa delle Liste (No Object dtype)

Pandas rappresenta liste come Python objects, causando inefficienze. Polars ha supporto nativo per il tipo `list`, permettendo operazioni vettorizzate sulle liste stesse.

In [13]:
# Liste in Polars: tipo nativo
pl.DataFrame({
    'ID': [1, 2, 3],
    'valori': [[1,2,3], [4,5], [6,7,8,9]]
})

ID,valori
i64,list[i64]
1,"[1, 2, 3]"
2,"[4, 5]"
3,"[6, 7, … 9]"


In [14]:
# Operazioni su liste — impossibili in Pandas senza apply()
df_list = pl.DataFrame({
    'ID': [1, 2, 3],
    'valori': [[1,2,3], [4,5], [6,7,8,9]]
})

df_list.with_columns([
    pl.col('valori').list.len().alias('lunghezza'),
    pl.col('valori').list.sum().alias('somma')
])

ID,valori,lunghezza,somma
i64,list[i64],u32,i64
1,"[1, 2, 3]",3,6
2,"[4, 5]",2,9
3,"[6, 7, … 9]",4,30


### 2. Lazy Evaluation

La **lazy evaluation** è uno dei vantaggi più importanti di Polars rispetto a Pandas.

#### Come funziona?

In modalità **eager** (come Pandas), ogni operazione viene eseguita immediatamente:
```python
df.filter(...)      # eseguito subito → produce un DataFrame intermedio
  .group_by(...)    # eseguito subito → produce un altro DataFrame intermedio  
  .sort(...)        # eseguito subito → produce il risultato finale
```

In modalità **lazy**, le operazioni vengono **registrate** ma **non eseguite**. Si costruisce un **piano di query** (query plan) che viene poi **ottimizzato** prima dell'esecuzione.

#### Cosa fa l'ottimizzatore?

Il query optimizer di Polars applica diverse tecniche automatiche:

- **Predicate pushdown**: i filtri vengono spostati il più "in basso" possibile nel piano, così i dati vengono filtrati prima di operazioni costose come aggregazioni o join
- **Projection pushdown**: se usi solo 3 colonne su 100, Polars legge dal disco solo quelle 3 — non carica l'intero file
- **Ottimizzazione delle espressioni**: operazioni ridondanti vengono eliminate, espressioni simili vengono accorpate
- **Parallelizzazione**: l'ottimizzatore decide come distribuire il lavoro su più core

Questo significa che Polars può essere **più veloce di codice scritto a mano**, perché l'ottimizzatore "vede" l'intera pipeline e prende decisioni globali.

In [15]:
# Eager: ogni passo produce un risultato intermedio
eager_result = (
    pl_agg
    .filter(pl.col('valore1') > 15)
    .group_by('gruppo')
    .agg([
        pl.col('valore1').sum().alias('valore1_sum'),
        pl.col('valore2').mean().alias('valore2_mean')
    ])
    .sort('valore1_sum', descending=True)
)

eager_result

gruppo,valore1_sum,valore2_mean
str,i64,f64
"""B""",55,205.0
"""C""",40,300.0
"""A""",20,120.0


In [16]:
# Lazy: costruisce il piano senza eseguirlo
lazy_query = (
    pl_agg.lazy()  # <-- attiva la modalità lazy
    .filter(pl.col('valore1') > 15)
    .group_by('gruppo')
    .agg([
        pl.col('valore1').sum().alias('valore1_sum'),
        pl.col('valore2').mean().alias('valore2_mean')
    ])
    .sort('valore1_sum', descending=True)
)

# Questo mostra il PIANO DI QUERY, non il risultato
lazy_query

Il diagramma mostrato è il **query plan non ottimizzato** (naive). Polars lo ottimizza internamente prima di eseguirlo.

In [17]:
# Per eseguire il piano e ottenere il risultato:
lazy_query.collect()

gruppo,valore1_sum,valore2_mean
str,i64,f64
"""B""",55,205.0
"""C""",40,300.0
"""A""",20,120.0


#### Lazy I/O: il vero punto di forza

La lazy evaluation diventa particolarmente potente con file su disco. Con `scan_csv` (invece di `read_csv`) Polars **non carica il file in memoria**: legge solo le colonne e le righe necessarie.

In [18]:
# Con scan_csv, Polars NON carica il file in memoria
df_lazy = pl.scan_csv("data.csv")

# Definisco una query complessa
result = (
    df_lazy
    .filter(pl.col('A') > 2)
    .select(['A', 'B'])
    .collect()  # Solo ora legge dal disco, e solo colonne A e B con A > 2
)

result

A,B
i64,i64
3,30
4,40
5,50


---

# 📖 Approfondimento Autonomo

Le sezioni seguenti contengono materiale aggiuntivo per lo studio individuale. Coprono operazioni più avanzate e alternative a Pandas che non vengono trattate a lezione ma che sono utili come riferimento.

## Pandas vs Polars: Trasformazione Colonne

In [19]:
# Pandas: aggiungere/modificare colonne
pandas_df_copy = pandas_df.copy()
pandas_df_copy['A'] = pandas_df_copy['A'] * 100
pandas_df_copy['B'] = pandas_df_copy['B'] + 5
pandas_df_copy['D'] = pandas_df['A'] * 100
pandas_df_copy

,A,B,C,D
0,100,15,a,100
1,200,25,b,200
2,300,35,c,300
3,400,45,d,400
4,500,55,e,500


In [20]:
# Polars: aggiungere/modificare colonne
polars_df.with_columns([
    pl.col("A") * 100,         # Sovrascrive "A"
    (pl.col("B") + 5).alias("B"),  # Sovrascrive "B"
    pl.col("A").mul(100).alias("D")  # Nuova colonna
])

A,B,C,D
i64,i64,str,i64
100,15,"""a""",100
200,25,"""b""",200
300,35,"""c""",300
400,45,"""d""",400
500,55,"""e""",500


## Fold (Riduzione personalizzata)

`pl.fold` permette di applicare una funzione di riduzione su più colonne, simile a `reduce` di Python.

In [21]:
# Fold: prodotto di più colonne
pl_agg.select(
    pl.fold(
        acc=pl.lit(1),
        function=lambda acc, x: acc * x,
        exprs=pl.col(['valore1', 'valore2'])
    ).alias('prodotto_valori')
)

prodotto_valori
i32
1000
2400
6000
5250
1650
12000


## Pandas vs Polars: Join

In [22]:
# Dataset per i join
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Dave']
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'customer_id': [1, 1, 2, 3, 1],
    'amount': [50, 25, 75, 100, 15]
})

In [23]:
# Pandas: merge (inner join di default)
pd.merge(customers, orders, on='customer_id')

,customer_id,name,order_id,amount
0,1,Alice,101,50
1,1,Alice,102,25
2,1,Alice,105,15
3,2,Bob,103,75
4,3,Charlie,104,100


In [24]:
# Polars: join esplicito + anti-join
pl_customers = pl.from_pandas(customers)
pl_orders = pl.from_pandas(orders)

# Anti-join: clienti SENZA ordini
pl_customers.join(
    pl_orders,
    on='customer_id',
    how='anti'
)

customer_id,name
i64,str
4,"""Dave"""


## Pandas vs Polars: Gestione Stringhe

In [25]:
# DataFrame con stringhe
text_df = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'text': ['hello world', 'data science', 'pandas vs polars', 'python is great']
})

# Manipolazione stringhe in Pandas
text_df['uppercase'] = text_df['text'].str.upper()
text_df['words'] = text_df['text'].str.split()
text_df['word_count'] = text_df['text'].str.count(' ') + 1

text_df

,id,text,uppercase,words,word_count
0,1,hello world,HELLO WORLD,"[hello, world]",2
1,2,data science,DATA SCIENCE,"[data, scie...",2
2,3,pandas vs p...,PANDAS VS P...,"[pandas, vs...",3
3,4,python is g...,PYTHON IS G...,"[python, is...",3


In [26]:
# DataFrame equivalente in Polars
pl_text = pl.DataFrame({
    'id': [1, 2, 3, 4],
    'text': ['hello world', 'data science', 'pandas vs polars', 'python is great']
})

# Manipolazione stringhe in Polars
pl_text.with_columns([
    pl.col('text').str.to_uppercase().alias('uppercase'),
    pl.col('text').str.split(' ').alias('words'),
    (pl.col('text').str.count_matches(' ') + 1).alias('word_count')
])

id,text,uppercase,words,word_count
i64,str,str,list[str],u32
1,"""hello world""","""HELLO WORLD""","[""hello"", ""world""]",2
2,"""data science""","""DATA SCIENCE""","[""data"", ""science""]",2
3,"""pandas vs polars""","""PANDAS VS POLARS""","[""pandas"", ""vs"", ""polars""]",3
4,"""python is great""","""PYTHON IS GREAT""","[""python"", ""is"", ""great""]",3


## Pipeline di Trasformazioni

Polars eccelle nella composizione di operazioni in pipeline fluide, sia in modalità eager che lazy.

In [27]:
# Esempio di pipeline in Polars (API Eager)
pl_result = (
    pl_agg
    .filter(pl.col('valore1') > 15)
    .group_by('gruppo')
    .agg([
        pl.col('valore1').sum().alias('valore1_sum'),
        pl.col('valore2').mean().alias('valore2_mean')
    ])
    .sort('valore1_sum', descending=True)
)

pl_result

gruppo,valore1_sum,valore2_mean
str,i64,f64
"""B""",55,205.0
"""C""",40,300.0
"""A""",20,120.0


## Ottimizzare Pandas

Se non puoi (o non vuoi) passare a Polars, ci sono tecniche per migliorare le prestazioni di Pandas:

1. **Selezionare solo le colonne necessarie** al caricamento con `usecols`
2. **Tipi di dato ottimali**: specificare `dtype` o usare `pd.Categorical` per dati categorici
3. **Chunking**: processare dati in blocchi con `chunksize`
4. **PyArrow**: utilizzare `engine='pyarrow'` per lettura/scrittura più veloce
5. **Vettorializzare**: preferire operazioni vettoriali invece di loop Python
6. **Query**: usare `df.query()` per espressioni complesse

In [28]:
# 1. Uso di usecols per selezionare solo colonne specifiche
# 2. Uso di dtype per specificare tipi esatti
# 3. Uso di engine='pyarrow' per prestazioni migliori

colonne = ['B', 'C']
tipi = {'B': 'int32', 'C': 'string'}

# Commentato per evitare l'esecuzione che consuma tempo
pd.read_csv('data.csv', 
            usecols=colonne, 
            dtype=tipi, 
            engine='pyarrow')

,B,C
0,10,a
1,20,b
2,30,c
3,40,d
4,50,e


In [29]:
# Esempio di caricamento con chunk per gestire dataset grandi

result_df = []
for i, chunk in enumerate(pd.read_csv("data.csv", chunksize=2)):
    print(f"Lunghezza del chunk # {i}: {len(chunk)}")
    chunk['A'] = chunk['A'].mean()
    
    result_df.append(chunk)
pd.concat(result_df, ignore_index=True)  # Unione dei chunk filtrati

Lunghezza del chunk # 0: 2
Lunghezza del chunk # 1: 2
Lunghezza del chunk # 2: 1


,A,B,C
0,1.5,10,a
1,1.5,20,b
2,3.5,30,c
3,3.5,40,d
4,5.0,50,e


## Alternative a Pandas

Oltre a Polars, diverse librerie sono state sviluppate per superare i limiti di Pandas:

**Dask**: estende Pandas e NumPy parallelizzando operazioni su cluster
- API quasi identica a Pandas (facile transizione)
- Elaborazione distribuita
- Gestione di dataset più grandi della memoria disponibile
- Integrazione con l'ecosistema scientifico

**cuDF (RAPIDS)**: implementazione GPU-accelerated di Pandas
- Accelerazione tramite GPU NVIDIA
- API simile a Pandas
- Integrabile con ecosistema RAPIDS per machine learning su GPU

**DuckDB**: motore di database analitico per DataFrame in-process
- Esecuzione di query SQL su DataFrame
- Performance fino a 10-100x più veloci di SQLite
- Ottimizzato per analisi OLAP
- Perfetta integrazione con Pandas e Apache Arrow